
# Telco Customer Churn — Hyperparameter Optimization

## Objective

This notebook performs hyperparameter optimization on the two best-performing models selected during model benchmarking:

- XGBoost — Standard
- Gradient Boosting — Standard

### Goals

1. Optimize model hyperparameters using Optuna (TPE).
2. Maximize Average Precision using Stratified 5-fold CV.
3. Track experiments and metrics using MLflow.
4. Compare optimized models against their baselines.
5. Select the best candidate for final evaluation.

### Methodology

- Reuse the stratified 80/20 train/test split exported during model benchmark.
- Perform hyperparameter optimization exclusively on training data.
- Fit preprocessing independently within each cross-validation training fold.
- Use the same Stratified 5-fold CV configuration as model benchmark.
- Preserve the held-out test set for final evaluation.
- Use fixed random seeds to improve reproducibility.



In [1]:

from pathlib import Path
import sys

import json

import numpy as np
import pandas as pd
import joblib

import optuna
import mlflow

from sklearn.base import clone
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

from xgboost import XGBClassifier

# Project root
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Optuna version:", optuna.__version__)
print("MLflow version:", mlflow.__version__)


Project root: C:\Users\Lenovo\Desktop\DS\Customer_Churn\Teleco_Customer_Churn
Optuna version: 5.1.0
MLflow version: 3.18.0



## 1. Load Training Objects

The training dataset, preprocessing configuration, and cross-validation strategy were prepared during model benchmarking and serialized using Joblib.

They are loaded directly to ensure consistency with the benchmarking phase.

This avoids repeating data cleaning, feature engineering, and train/test splitting.

The preprocessing configuration is reused from modeling. Within each cross-validation fold, a fresh pipeline is fitted exclusively on the corresponding training partition.


In [2]:
bundle_path = (
    project_root
    / "data"
    / "processed"
    / "training_bundle.joblib"
)

bundle = joblib.load(bundle_path)

X_train = bundle["X_train"]
y_train = bundle["y_train"]
preprocessor_tree = bundle["preprocessor_tree"]
cv = bundle["cv"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("CV folds:", cv.get_n_splits())
print("Preprocessor:", type(preprocessor_tree).__name__)

print("\nTarget distribution:")
print(y_train.value_counts(normalize=True))


X_train: (5634, 19)
y_train: (5634,)
CV folds: 5
Preprocessor: ColumnTransformer

Target distribution:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64



## 2. XGBoost Hyperparameter Optimization

### Objective Function

Optuna is used to search for effective XGBoost hyperparameter configurations.

For each trial:

1. Suggest a combination of hyperparameters.
2. Initialize an XGBoost classifier.
3. Construct a preprocessing and classification pipeline.
4. Evaluate the pipeline using Stratified 5-fold CV.
5. Compute the mean and standard deviation of Average Precision.
6. Return the mean Average Precision to Optuna.

The optimization maximizes cross-validated Average Precision.

Additional evaluation metrics are recorded for analysis.


In [3]:

def objective_xgb(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 500, step=50
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 2, 8
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        )
    }

    # 2. Initialize XGBoost
    model = XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    # 3. Build preprocessing + model pipeline
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor_tree)),
        ("model", model)
    ])

    # 4. Stratified 5-fold CV
    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "average_precision": "average_precision",
            "roc_auc": "roc_auc"
        },
        n_jobs=1,
        error_score="raise"
    )

    # 5. Aggregate CV metrics
    ap_scores = cv_results["test_average_precision"]
    roc_scores = cv_results["test_roc_auc"]

    mean_ap = ap_scores.mean()
    std_ap = ap_scores.std()
    mean_roc_auc = roc_scores.mean()

    # 6. Store additional metrics
    trial.set_user_attr("std_ap", float(std_ap))
    trial.set_user_attr("mean_roc_auc", float(mean_roc_auc))

    # 7. Return optimization objective
    return float(mean_ap)



## 3. MLflow Experiment Tracking

MLflow is used to track and organize the hyperparameter optimization experiments.

For each Optuna trial, we record:

- Suggested hyperparameters.
- Mean cross-validated Average Precision.
- Standard deviation of Average Precision.
- Mean cross-validated ROC-AUC.
- Model and optimization metadata.

Experiments are stored locally in a SQLite database to support traceability and reproducibility.


In [4]:

# Local MLflow tracking database
mlflow_db = project_root / "mlflow.db"

mlflow.set_tracking_uri(
    f"sqlite:///{mlflow_db.as_posix()}"
)

# Create or select experiment
mlflow.set_experiment(
    "Telco_Churn_Hyperparameter_Tuning"
)

print("Tracking URI:", mlflow.get_tracking_uri())


Tracking URI: sqlite:///C:/Users/Lenovo/Desktop/DS/Customer_Churn/Teleco_Customer_Churn/mlflow.db



## 4. XGBoost Hyperparameter Tuning

### Optuna–MLflow Integration

The existing `objective_xgb()` function performs model training and cross-validation.

A tracking wrapper is introduced to log each Optuna trial as an individual MLflow run.

This separates model evaluation from experiment tracking and allows every hyperparameter configuration to be inspected independently.


In [5]:

def objective_xgb_mlflow(trial):

    with mlflow.start_run(
        run_name=f"XGBoost_Trial_{trial.number:03d}"
    ):

        mean_ap = objective_xgb(trial)

        mlflow.log_params(trial.params)

        mlflow.log_metrics({
            "cv_ap_mean": mean_ap,
            "cv_ap_std": trial.user_attrs["std_ap"],
            "cv_roc_auc_mean": trial.user_attrs["mean_roc_auc"]
        })

        mlflow.set_tags({
            "model_family": "XGBoost",
            "stage": "official_tuning",
            "optimizer": "Optuna_TPE",
            "cv_strategy": "Stratified_5Fold"
        })

    return mean_ap



### Optuna Study Configuration

The XGBoost hyperparameter search uses:

- TPE sampler with a fixed random seed.
- 50 optimization trials.
- 10 initial random exploration trials.
- Mean CV Average Precision as the maximization objective.
- Stratified 5-fold cross-validation.

The study is stored in SQLite to preserve completed trials and support resuming interrupted optimization.

Each trial is tracked separately in MLflow.


In [6]:

# Persistent Optuna storage
optuna_db = project_root / "optuna_studies.db"

storage_uri = f"sqlite:///{optuna_db.as_posix()}"

# Create or resume study
study_xgb = optuna.create_study(
    study_name="Telco_XGBoost_Tuning",
    storage=storage_uri,
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=10
    )
)

# Complete up to 50 trials
remaining_trials = max(
    0,
    50 - len(study_xgb.trials)
)

if remaining_trials > 0:
    study_xgb.optimize(
        objective_xgb_mlflow,
        n_trials=remaining_trials
    )

print("Total trials:", len(study_xgb.trials))
print("Best trial:", study_xgb.best_trial.number)
print("Best CV AP:", study_xgb.best_value)

print("\nBest hyperparameters:")
for name, value in study_xgb.best_params.items():
    print(f"{name}: {value}")


[I 2026-11-18 00:05:20,557] Using an existing study with `study_name='Telco_XGBoost_Tuning'` instead of creating a new one.


Total trials: 50
Best trial: 43
Best CV AP: 0.6742037800267484

Best hyperparameters:
n_estimators: 150
max_depth: 2
learning_rate: 0.08315510161972098
subsample: 0.7996257787814657
colsample_bytree: 0.6353718081178203
min_child_weight: 3



### XGBoost Optimization Results

The completed Optuna study is analyzed to identify the best hyperparameter configuration.

In [7]:

# Collect completed trials
xgb_results = pd.DataFrame([
    {
        "trial": trial.number,
        "cv_ap_mean": trial.value,
        "cv_ap_std": trial.user_attrs.get("std_ap"),
        "cv_roc_auc_mean": trial.user_attrs.get("mean_roc_auc"),
        **trial.params
    }
    for trial in study_xgb.trials
    if trial.state == optuna.trial.TrialState.COMPLETE
])

# Sort by primary optimization metric
xgb_results = xgb_results.sort_values(
    "cv_ap_mean",
    ascending=False
).reset_index(drop=True)

display(xgb_results.head(10))


,trial,cv_ap_mean,cv_ap_std,cv_roc_auc_mean,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight
0,43,0.674204,0.021250,0.850892,150,2,0.083155,0.799626,0.635372,3
1,49,0.673622,0.020230,0.849884,150,2,0.094519,0.810549,0.606075,4
2,29,0.673293,0.020708,0.849603,150,3,0.061270,0.803637,0.714704,2
3,31,0.672910,0.018464,0.849052,150,3,0.073758,0.831149,0.716589,6
4,32,0.672545,0.020789,0.849951,200,2,0.069988,0.804090,0.635602,5
5,6,0.671984,0.020373,0.849536,200,2,0.077662,0.776061,0.648815,5
6,39,0.671853,0.019789,0.849830,200,2,0.075361,0.820986,0.624545,3
7,37,0.671843,0.019262,0.848396,250,3,0.048249,0.807470,0.699166,6
8,30,0.671809,0.020227,0.848780,150,3,0.075638,0.874523,0.689505,4
9,2,0.671653,0.021132,0.850007,450,3,0.017241,0.673362,0.721697,6


In [8]:

baseline_ap = 0.6713

best_tuned_ap = study_xgb.best_value

improvement = best_tuned_ap - baseline_ap

print(f"Baseline AP: {baseline_ap:.4f}")
print(f"Tuned AP:    {best_tuned_ap:.4f}")
print(f"Difference:  {improvement:+.4f}")


Baseline AP: 0.6713
Tuned AP:    0.6742
Difference:  +0.0029



### XGBoost Optimization — Summary

Optuna completed 50 trials using TPE and Stratified 5-fold cross-validation.

The best configuration was identified in trial 43, achieving:

- Mean CV Average Precision: 0.6742
- Standard deviation: 0.0213
- Mean CV ROC-AUC: 0.8509

Compared with the baseline (AP = 0.6713), tuning produced a small absolute improvement of approximately 0.0029.

The strongest configurations generally favored shallow trees (max_depth of 2–3), moderate learning rates, and subsampling.

Performance variability remained similar to the baseline. Because the best configuration was selected using the optimization folds, the observed improvement should be interpreted cautiously.

The tuned XGBoost configuration is retained for comparison with the tuned Gradient Boosting model.



## 5. Gradient Boosting Hyperparameter Optimization

### Objective Function

Gradient Boosting is optimized using Optuna with the same evaluation protocol as XGBoost.

Each trial proposes a hyperparameter configuration, builds a preprocessing and classification pipeline, and evaluates it through Stratified 5-fold CV.

Mean Average Precision is maximized, while ROC-AUC and Average Precision standard deviation are recorded for additional analysis.


In [9]:

def objective_gb(trial):

    # 1. Hyperparameter search space
    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 400, step=50
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 1, 5
        ),
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf", 1, 20
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),
        "max_features": trial.suggest_float(
            "max_features", 0.6, 1.0
        )
    }

    # 2. Initialize Gradient Boosting
    model = GradientBoostingClassifier(
        **params,
        random_state=42
    )

    # 3. Build preprocessing + model pipeline
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor_tree)),
        ("model", model)
    ])

    # 4. Stratified 5-fold CV
    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "average_precision": "average_precision",
            "roc_auc": "roc_auc"
        },
        n_jobs=1,
        error_score="raise"
    )

    # 5. Aggregate metrics
    ap_scores = cv_results["test_average_precision"]
    roc_scores = cv_results["test_roc_auc"]

    mean_ap = ap_scores.mean()
    std_ap = ap_scores.std()
    mean_roc_auc = roc_scores.mean()

    # 6. Store additional metrics
    trial.set_user_attr("std_ap", float(std_ap))
    trial.set_user_attr("mean_roc_auc", float(mean_roc_auc))

    return float(mean_ap)



### MLflow Integration

Each Gradient Boosting trial is tracked as an individual MLflow run.

The same experiment is used as for XGBoost, allowing consistent comparison between model families.

Each run records hyperparameters, cross-validation metrics, and model metadata.


In [10]:

def objective_gb_mlflow(trial):

    with mlflow.start_run(
        run_name=f"GradientBoosting_Trial_{trial.number:03d}"
    ):

        mean_ap = objective_gb(trial)

        # Log hyperparameters
        mlflow.log_params(trial.params)

        # Log evaluation metrics
        mlflow.log_metrics({
            "cv_ap_mean": mean_ap,
            "cv_ap_std": trial.user_attrs["std_ap"],
            "cv_roc_auc_mean": trial.user_attrs["mean_roc_auc"]
        })

        # Log metadata
        mlflow.set_tags({
            "model_family": "GradientBoosting",
            "stage": "official_tuning",
            "optimizer": "Optuna_TPE",
            "cv_strategy": "Stratified_5Fold"
        })

    return mean_ap



### Full Gradient Boosting Optimization

The Gradient Boosting search uses 50 Optuna trials with the TPE sampler.

The study is stored persistently in SQLite, while individual trial results are logged to MLflow.

The optimization maximizes mean cross-validated Average Precision.


In [11]:

study_gb = optuna.create_study(
    study_name="Telco_GradientBoosting_Tuning",
    storage=storage_uri,
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=10
    )
)

remaining_trials = max(
    0,
    50 - len(study_gb.trials)
)

if remaining_trials > 0:
    study_gb.optimize(
        objective_gb_mlflow,
        n_trials=remaining_trials
    )

print("Total trials:", len(study_gb.trials))
print("Best trial:", study_gb.best_trial.number)
print("Best CV AP:", study_gb.best_value)

print("\nBest hyperparameters:")
for name, value in study_gb.best_params.items():
    print(f"{name}: {value}")


[I 2026-11-18 00:05:21,019] Using an existing study with `study_name='Telco_GradientBoosting_Tuning'` instead of creating a new one.


Total trials: 50
Best trial: 29
Best CV AP: 0.6733800688987557

Best hyperparameters:
n_estimators: 150
learning_rate: 0.02655331725728516
max_depth: 4
min_samples_leaf: 1
subsample: 0.7103454500437947
max_features: 0.7384072304692317


In [12]:

gb_results = pd.DataFrame([
    {
        "trial": trial.number,
        "cv_ap_mean": trial.value,
        "cv_ap_std": trial.user_attrs.get("std_ap"),
        "cv_roc_auc_mean": trial.user_attrs.get("mean_roc_auc"),
        **trial.params
    }
    for trial in study_gb.trials
    if trial.state == optuna.trial.TrialState.COMPLETE
])

gb_results = gb_results.sort_values(
    "cv_ap_mean",
    ascending=False
).reset_index(drop=True)

display(gb_results.head(10))


,trial,cv_ap_mean,cv_ap_std,cv_roc_auc_mean,n_estimators,learning_rate,max_depth,min_samples_leaf,subsample,max_features
0,29,0.673380,0.016993,0.851085,150,0.026553,4,1,0.710345,0.738407
1,27,0.673345,0.018769,0.850994,200,0.027987,4,2,0.749035,0.753152
2,41,0.673232,0.018980,0.850751,250,0.018049,4,3,0.688390,0.707219
3,37,0.673152,0.020545,0.850737,200,0.019207,4,4,0.689312,0.810398
4,3,0.672599,0.016918,0.849814,250,0.023928,4,3,0.716858,0.746545
5,26,0.672439,0.021842,0.850886,200,0.017003,4,2,0.676032,0.739322
6,47,0.672422,0.017325,0.849870,250,0.021034,4,4,0.703996,0.683702
7,20,0.672288,0.018693,0.850640,200,0.014628,4,3,0.706156,0.626845
8,34,0.672215,0.019166,0.850613,200,0.020538,4,1,0.718581,0.754091
9,39,0.672113,0.020004,0.850191,200,0.016752,4,6,0.715309,0.807379


In [13]:

baseline_gb_ap = 0.6681

best_gb_ap = study_gb.best_value

print(f"Baseline AP: {baseline_gb_ap:.4f}")
print(f"Tuned AP:    {best_gb_ap:.4f}")
print(f"Difference:  {best_gb_ap - baseline_gb_ap:+.4f}")


Baseline AP: 0.6681
Tuned AP:    0.6734
Difference:  +0.0053



### Gradient Boosting Optimization — Summary

Optuna completed 50 trials using TPE and Stratified 5-fold cross-validation.

The best configuration was identified in trial 29:

- Mean CV Average Precision: 0.6734
- Standard deviation: 0.0170
- Mean CV ROC-AUC: 0.8511

Compared with the baseline (AP = 0.6681), optimization improved mean AP by approximately 0.0053.

The best-performing configurations consistently favored a maximum tree depth of 4, relatively low learning rates, and subsampling.

The tuned configuration also exhibited lower cross-validation variability than the baseline.

These findings are based on the optimization folds and do not establish an improvement on unseen data.



## 6. Final Model Comparison

The optimized XGBoost and Gradient Boosting models are compared against their baselines.

The primary selection criterion is mean cross-validated Average Precision.

ROC-AUC and the standard deviation of Average Precision are considered complementary indicators.

Since hyperparameter selection was performed using the same cross-validation folds, the best observed CV scores may be optimistic.


In [14]:

comparison_df = pd.DataFrame([
    {
        "model": "XGBoost",
        "baseline_ap": 0.6713,
        "tuned_ap": study_xgb.best_value,
        "cv_ap_std": study_xgb.best_trial.user_attrs["std_ap"],
        "cv_roc_auc": study_xgb.best_trial.user_attrs["mean_roc_auc"],
        "best_trial": study_xgb.best_trial.number
    },
    {
        "model": "Gradient Boosting",
        "baseline_ap": 0.6681,
        "tuned_ap": study_gb.best_value,
        "cv_ap_std": study_gb.best_trial.user_attrs["std_ap"],
        "cv_roc_auc": study_gb.best_trial.user_attrs["mean_roc_auc"],
        "best_trial": study_gb.best_trial.number
    }
])

comparison_df["ap_improvement"] = (
    comparison_df["tuned_ap"]
    - comparison_df["baseline_ap"]
)

comparison_df = comparison_df.sort_values(
    "tuned_ap",
    ascending=False
).reset_index(drop=True)

display(comparison_df.round(4))


,model,baseline_ap,tuned_ap,cv_ap_std,cv_roc_auc,best_trial,ap_improvement
0,XGBoost,0.6713,0.6742,0.0213,0.8509,43,0.0029
1,Gradient Boosting,0.6681,0.6734,0.0170,0.8511,29,0.0053


In [15]:


results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)


# Export all completed optimization trials
xgb_results.to_csv(
    results_dir / "05_xgboost_trials.csv",
    index=False
)

gb_results.to_csv(
    results_dir / "05_gradient_boosting_trials.csv",
    index=False
)

print("Trial results exported successfully.")

# Export comparison table
comparison_df.to_csv(
    results_dir / "05_model_comparison.csv",
    index=False
)

# Export best hyperparameters
best_params = {
    "XGBoost": study_xgb.best_params,
    "GradientBoosting": study_gb.best_params
}

with open(
    results_dir / "05_best_params.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(best_params, f, indent=4)

print("Results exported successfully.")


Trial results exported successfully.
Results exported successfully.


In [16]:

# Full configurations for Day 6
best_model_configs = {
    "XGBoost": {
        **study_xgb.best_params,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1
    },
    "GradientBoosting": {
        **study_gb.best_params,
        "random_state": 42
    }
}

with open(
    results_dir / "05_best_model_configs.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(best_model_configs, f, indent=4)

print("Full model configurations exported.")


Full model configurations exported.



## 7. Final Summary

Hyperparameter optimization was completed for the two leading model families using Optuna TPE, with 50 trials per model.

All configurations were evaluated using the same Stratified 5-fold cross-validation strategy, maximizing Average Precision. Experiments were tracked using MLflow.

### Key Findings

- XGBoost achieved the highest mean CV Average Precision (0.6742 ± 0.0213).
- Gradient Boosting achieved a very similar mean CV AP (0.6734 ± 0.0170).
- Both optimized configurations slightly outperformed their respective baselines.
- Gradient Boosting exhibited lower variability in cross-validation scores.
- The observed difference in Average Precision between the two tuned models was only 0.0008.

### Model Selection

XGBoost is retained as the primary candidate because it achieved the highest cross-validated Average Precision, our predefined primary metric.

Gradient Boosting remains a competitive alternative due to its comparable AP and lower cross-validation variability.

The small performance difference does not establish statistically meaningful superiority.

### Next Steps

- Generate out-of-fold predictions using the selected training configuration.
- Optimize the decision threshold according to predictive and business objectives.
- Evaluate the finalized model once on the held-out test set.
- Save the final model pipeline and associated metadata using MLflow.
